In [1]:
# ============================================================================
# NCAA March Madness 2026 — ELITE ENSEMBLE PREDICTOR (CORRECTED)
# ============================================================================

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'optuna', '-q'], check=False)

import os, glob, warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

DATA_PATH = '/kaggle/input/competitions/march-machine-learning-mania-2026'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


# ══════════════════════════════════════════════════════════════════════════════
#  0.  LOAD DATA WITH CORRECT FILE NAMES
# ══════════════════════════════════════════════════════════════════════════════
def load_all_data(data_path):
    """Load all required CSV files."""
    print("=" * 80)
    print("  LOADING DATA")
    print("=" * 80 + "\n")
    
    files_to_load = {
        'MNCAATourneySeeds': 'MNCAATourneySeeds',
        'WNCAATourneySeeds': 'WNCAATourneySeeds',
        'MRegularSeasonDetailedResults': 'MRegularSeasonDetailedResults',
        'WRegularSeasonDetailedResults': 'WRegularSeasonDetailedResults',
        'MNCAATourneyDetailedResults': 'MNCAATourneyDetailedResults',
        'WNCAATourneyDetailedResults': 'WNCAATourneyDetailedResults',
        'SampleSubmissionStage2': 'SampleSubmissionStage2',
        'SampleSubmissionStage1': 'SampleSubmissionStage1',
    }
    
    data = {}
    for key, fname in files_to_load.items():
        fpath = os.path.join(data_path, f'{fname}.csv')
        if os.path.exists(fpath):
            df = pd.read_csv(fpath, encoding='latin-1')
            data[key] = df
            print(f"  ✓ {fname:<45s} ({len(df):8,} rows)")
        else:
            print(f"  ✗ {fname:<45s} NOT FOUND")
    
    print(f"\n  Loaded {len(data)} files\n")
    return data


# ══════════════════════════════════════════════════════════════════════════════
#  1.  ELO RATING
# ══════════════════════════════════════════════════════════════════════════════
def compute_elo(games_df, k=20, initial=1500, carry_pct=0.75):
    """Season-aware Elo with cross-season regression to mean."""
    elo, records, prev_season = {}, [], None
    for row in games_df.sort_values(['Season', 'DayNum']).itertuples():
        season = int(row.Season)
        if prev_season is not None and season != prev_season:
            for key in list(elo.keys()):
                elo[key] = initial + carry_pct * (elo[key] - initial)
        prev_season = season
        w, l = int(row.WTeamID), int(row.LTeamID)
        elo.setdefault((season, w), initial)
        elo.setdefault((season, l), initial)
        ew  = 1 / (1 + 10 ** ((elo[(season, l)] - elo[(season, w)]) / 400))
        mov = np.log(abs(row.WScore - row.LScore) + 1) * (2.2 / (ew * 0.001 + 1.0))
        d   = k * mov * (1 - ew)
        elo[(season, w)] += d
        elo[(season, l)] -= d
        records += [
            (season, w, row.DayNum, elo[(season, w)]),
            (season, l, row.DayNum, elo[(season, l)])
        ]
    df = pd.DataFrame(records, columns=['Season', 'TeamID', 'DayNum', 'Elo'])
    return (df.sort_values('DayNum')
              .groupby(['Season', 'TeamID'])['Elo'].last()
              .reset_index().rename(columns={'Elo': 'FinalElo'}))


# ══════════════════════════════════════════════════════════════════════════════
#  2.  ADVANCED TEAM STATISTICS
# ══════════════════════════════════════════════════════════════════════════════
def compute_team_stats(games_df, last_n=14):
    """Compute full-season and recent-form advanced statistics."""
    e = 1e-6

    def one_side(df, p, op):
        return pd.DataFrame({
            'Season':  df['Season'].values,
            'DayNum':  df['DayNum'].values,
            'TeamID':  df[f'{p}TeamID'].values,
            'Score':   df[f'{p}Score'].values,
            'OppScore':df[f'{op}Score'].values,
            'FGM':     df[f'{p}FGM'].values,
            'FGA':     df[f'{p}FGA'].values,
            'FGM3':    df[f'{p}FGM3'].values,
            'FGA3':    df[f'{p}FGA3'].values,
            'FTM':     df[f'{p}FTM'].values,
            'FTA':     df[f'{p}FTA'].values,
            'OR':      df[f'{p}OR'].values,
            'DR':      df[f'{p}DR'].values,
            'Ast':     df[f'{p}Ast'].values,
            'TO':      df[f'{p}TO'].values,
            'Stl':     df[f'{p}Stl'].values,
            'Blk':     df[f'{p}Blk'].values,
            'PF':      df[f'{p}PF'].values,
            'OppFGM':  df[f'{op}FGM'].values,
            'OppFGA':  df[f'{op}FGA'].values,
            'OppFGM3': df[f'{op}FGM3'].values,
            'OppOR':   df[f'{op}OR'].values,
            'OppTO':   df[f'{op}TO'].values,
            'Win':     float(p == 'W'),
        })

    df = pd.concat([one_side(games_df, 'W', 'L'),
                    one_side(games_df, 'L', 'W')], ignore_index=True)

    df['Poss']   = df['FGA'] - df['OR'] + df['TO'] + 0.44 * df['FTA']
    df['OPoss']  = df['OppFGA'] - df['OppOR'] + df['OppTO'] + e
    df['eFG']    = (df['FGM'] + 0.5 * df['FGM3']) / (df['FGA'] + e)
    df['TS']     = df['Score'] / (2 * (df['FGA'] + 0.44 * df['FTA']) + e)
    df['3PAR']   = df['FGA3'] / (df['FGA'] + e)
    df['FTR']    = df['FTA']  / (df['FGA'] + e)
    df['AstR']   = df['Ast']  / (df['Poss'] + e)
    df['TOVR']   = df['TO']   / (df['Poss'] + e)
    df['ORR']    = df['OR']   / (df['OR'] + df['OppFGA'] - df['OppFGM'] + e)
    df['DRR']    = df['DR']   / (df['DR'] + df['OppOR'] + e)
    df['OffRtg'] = df['Score']     / (df['Poss'] + e) * 100
    df['DefRtg'] = df['OppScore']  / (df['Poss'] + e) * 100
    df['NetRtg'] = df['OffRtg'] - df['DefRtg']
    df['Pyth']   = df['Score'] ** 11.5 / (df['Score'] ** 11.5 + df['OppScore'] ** 11.5 + e)
    df['Margin'] = df['Score'] - df['OppScore']
    df['FG2Pct'] = (df['FGM'] - df['FGM3']) / (df['FGA'] - df['FGA3'] + e)
    df['FG3Pct'] = df['FGM3'] / (df['FGA3'] + e)
    df['FTPct']  = df['FTM']  / (df['FTA'] + e)
    df['BlkR']   = df['Blk']  / (df['OppFGA'] - df['OppFGM3'] + e)
    df['StlR']   = df['Stl']  / (df['OPoss'] + e)
    df['ClutchWin'] = ((df['Win'] == 1) & (df['Margin'].abs() <= 5)).astype(float)

    df = df.sort_values(['Season', 'TeamID', 'DayNum'])

    STATS = ['Win', 'Score', 'OppScore', 'Margin', 'eFG', 'TS', '3PAR', 'FTR',
             'AstR', 'TOVR', 'ORR', 'DRR', 'OffRtg', 'DefRtg', 'NetRtg',
             'Pyth', 'FG2Pct', 'FG3Pct', 'FTPct', 'BlkR', 'StlR', 'ClutchWin']

    full = (df.groupby(['Season', 'TeamID'])[STATS]
              .agg(['mean', 'std'])
              .reset_index())
    full.columns = ['Season', 'TeamID'] + [f'{c}_{a}' for c, a in full.columns[2:]]

    rec = (df.groupby(['Season', 'TeamID']).tail(last_n)
             .groupby(['Season', 'TeamID'])
             [['Win', 'NetRtg', 'Margin', 'eFG', 'OffRtg', 'DefRtg', 'TOVR', 'ORR', 'Pyth']]
             .mean().reset_index()
             .rename(columns={c: f'Rec_{c}' for c in
                               ['Win', 'NetRtg', 'Margin', 'eFG', 'OffRtg', 'DefRtg', 'TOVR', 'ORR', 'Pyth']}))

    opp_def = df.groupby(['Season', 'TeamID'])['DefRtg'].mean().reset_index()
    opp_def.columns = ['Season', 'OppID', 'OppDefRtg']
    opp_map = pd.concat([
        games_df[['Season', 'WTeamID', 'LTeamID']].rename(
            columns={'WTeamID': 'TeamID', 'LTeamID': 'OppID'}),
        games_df[['Season', 'WTeamID', 'LTeamID']].rename(
            columns={'LTeamID': 'TeamID', 'WTeamID': 'OppID'}),
    ]).merge(opp_def, on=['Season', 'OppID'], how='left')
    sos = (opp_map.groupby(['Season', 'TeamID'])['OppDefRtg']
                  .mean().reset_index()
                  .rename(columns={'OppDefRtg': 'SOS_OppDefRtg'}))

    stats = full.merge(rec, on=['Season', 'TeamID'], how='left')
    stats = stats.merge(sos, on=['Season', 'TeamID'], how='left')
    return stats


# ══════════════════════════════════════════════════════════════════════════════
#  3.  HEAD-TO-HEAD HISTORY
# ══════════════════════════════════════════════════════════════════════════════
def compute_h2h(tourney_games):
    """Compute H2H records with Bayesian shrinkage toward 0.5."""
    df = tourney_games.copy()
    df['T1']    = df[['WTeamID', 'LTeamID']].min(axis=1)
    df['T2']    = df[['WTeamID', 'LTeamID']].max(axis=1)
    df['T1Win'] = (df['WTeamID'] == df['T1']).astype(float)
    h2h = (df.groupby(['T1', 'T2'])['T1Win']
             .agg(H2H_Games='count', H2H_WinRate='mean')
             .reset_index())
    prior_n = 3
    h2h['H2H_WinRate'] = (h2h['H2H_Games'] * h2h['H2H_WinRate'] + prior_n * 0.5) / \
                          (h2h['H2H_Games'] + prior_n)
    h2h['H2H_Games']   = h2h['H2H_Games'].clip(upper=5)
    return h2h


# ══════════════════════════════════════════════════════════════════════════════
#  4.  BUILD MATCHUP FEATURES
# ══════════════════════════════════════════════════════════════════════════════
def build_features(matchups, seeds, elo_df, stats_df, is_train=True):
    """Construct comprehensive matchup features."""
    df = matchups.copy()
    
    if is_train:
        df['T1']    = df[['WTeamID', 'LTeamID']].min(axis=1)
        df['T2']    = df[['WTeamID', 'LTeamID']].max(axis=1)
        df['Label'] = (df['WTeamID'] == df['T1']).astype(float)
    else:
        # Parse ID: "2026_1101_1102"
        df[['Season_str', 'T1_str', 'T2_str']] = df['ID'].str.split('_', expand=True)
        df['T1'] = df['T1_str'].astype(int)
        df['T2'] = df['T2_str'].astype(int)
        df['Season'] = df['Season_str'].astype(int)
    
    df['Season'] = df['Season'].astype(int)

    # Seed Features
    df['Seed1']       = df.apply(lambda r: seeds.get(f"{r['Season']}_{r['T1']}", 16), axis=1)
    df['Seed2']       = df.apply(lambda r: seeds.get(f"{r['Season']}_{r['T2']}", 16), axis=1)
    df['SeedDiff']    = df['Seed1'] - df['Seed2']
    df['SeedSum']     = df['Seed1'] + df['Seed2']
    df['SeedRatio']   = df['Seed1'] / (df['Seed2'] + 1e-3)
    df['SeedWinProb'] = 1 / (1 + np.exp(0.4 * df['SeedDiff']))
    df['UpsetPot']    = (df['Seed1'] > df['Seed2']).astype(float)

    # Elo Features
    df = df.merge(elo_df.rename(columns={'TeamID': 'T1', 'FinalElo': 'Elo1'}),
                  on=['Season', 'T1'], how='left')
    df = df.merge(elo_df.rename(columns={'TeamID': 'T2', 'FinalElo': 'Elo2'}),
                  on=['Season', 'T2'], how='left')
    df['EloDiff']    = df['Elo1'] - df['Elo2']
    df['EloWinProb'] = 1 / (1 + 10 ** (-df['EloDiff'] / 400))
    df['EloRatio']   = df['Elo1'] / (df['Elo2'] + 1e-3)

    # Team Stats
    sc = [c for c in stats_df.columns if c not in ['Season', 'TeamID']]
    df = df.merge(stats_df.rename(columns={'TeamID': 'T1', **{c: f'T1_{c}' for c in sc}}),
                  on=['Season', 'T1'], how='left')
    df = df.merge(stats_df.rename(columns={'TeamID': 'T2', **{c: f'T2_{c}' for c in sc}}),
                  on=['Season', 'T2'], how='left')
    for c in sc:
        df[f'D_{c}'] = df[f'T1_{c}'] - df[f'T2_{c}']
        if 'Rtg' in c or c in ['eFG_mean', 'NetRtg_mean', 'Pyth_mean']:
            df[f'R_{c}'] = df[f'T1_{c}'] / (df[f'T2_{c}'].abs() + 1e-3)

    # Interaction Features
    df['Elo_x_SeedDiff']     = df['EloDiff']       * df['SeedDiff']
    df['NetRtg_x_SeedDiff']  = df['D_NetRtg_mean'] * df['SeedDiff']
    df['Elo_x_NetRtg']       = df['EloDiff']       * df['D_NetRtg_mean']
    df['Seed_x_Pyth']        = df['SeedDiff']      * df['D_Pyth_mean']
    df['OffDef_Interaction'] = df['D_OffRtg_mean'] * (-df['D_DefRtg_mean'])

    return df


# ══════════════════════════════════════════════════════════════════════════════
#  5.  LEAKAGE-SAFE FEATURE SELECTION
# ══════════════════════════════════════════════════════════════════════════════
def select_features(X, y, feature_names, seasons, n_top=80, n_shadow_runs=5,
                    held_out_seasons=5):
    """Select features only on early seasons to avoid leakage."""
    unique_s = sorted(np.unique(seasons))
    safe_mask = np.isin(seasons, unique_s[:-held_out_seasons])
    Xs, ys = X[safe_mask], y[safe_mask]

    print(f"  Feature selection on {safe_mask.sum()} games "
          f"({len(unique_s) - held_out_seasons} early seasons)")

    probe = lgb.LGBMClassifier(n_estimators=300, num_leaves=31, max_depth=5,
                                learning_rate=0.05, random_state=RANDOM_SEED, verbose=-1)
    probe.fit(Xs, ys)
    imp = pd.Series(probe.feature_importances_, index=feature_names).sort_values(ascending=False)
    top_names = imp.head(n_top).index.tolist()

    X_top = Xs[:, [feature_names.index(f) for f in top_names]]
    shadow_scores = np.zeros(len(top_names))
    for _ in range(n_shadow_runs):
        X_shadow = X_top.copy()
        for j in range(X_shadow.shape[1]):
            X_shadow[:, j] = np.random.permutation(X_shadow[:, j])
        X_aug = np.hstack([X_top, X_shadow])
        probe2 = lgb.LGBMClassifier(n_estimators=200, num_leaves=15, max_depth=4,
                                     learning_rate=0.05, random_state=RANDOM_SEED, verbose=-1)
        probe2.fit(X_aug, ys)
        imp2 = probe2.feature_importances_
        shadow_scores += (imp2[:len(top_names)] > imp2[len(top_names):].max())

    keep_mask = shadow_scores >= (n_shadow_runs * 0.4)
    selected  = [n for n, k in zip(top_names, keep_mask) if k]

    must_keep = ['SeedDiff', 'EloDiff', 'EloWinProb', 'SeedWinProb',
                 'D_NetRtg_mean', 'D_OffRtg_mean', 'D_DefRtg_mean',
                 'D_Pyth_mean', 'D_Win_mean']
    for f in must_keep:
        if f in feature_names and f not in selected:
            selected.append(f)

    print(f"  Selected {len(selected)} features (from {X.shape[1]})")
    return selected


# ═════════���════════════════════════════════════════════════════════════════════
#  6.  TEMPORAL CV HELPER
# ══════════════════════════════════════════════════════════════════════════════
def temporal_cv_score(model, X, y, seasons, min_train=80):
    """Score model using temporal CV (train < test season)."""
    unique = sorted(np.unique(seasons))
    preds, trues = [], []
    for ts in unique:
        tr = np.where(seasons < ts)[0]
        te = np.where(seasons == ts)[0]
        if len(tr) < min_train or len(te) == 0:
            continue
        model.fit(X[tr], y[tr])
        preds.append(model.predict_proba(X[te])[:, 1])
        trues.append(y[te])
    if not preds:
        return 1.0
    return log_loss(np.concatenate(trues), np.concatenate(preds))


# ══════════════════════════════════════════════════════════════════════════════
#  7.  BAYESIAN TUNING
# ══════════════════════════════════════════════════════════════════════════════
def _run_study(objective_fn, n_trials, label):
    sampler = TPESampler(seed=RANDOM_SEED, n_startup_trials=10, multivariate=True)
    study   = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(objective_fn, n_trials=n_trials, show_progress_bar=False)
    best = study.best_trial
    print(f"  ✓ {label:<12s}  LL={best.value:.5f}  ({len(study.trials)} trials)")
    return best.params, best.value


def tune_lgb(X, y, seasons, n_trials=60):
    def obj(trial):
        p = dict(
            num_leaves        = trial.suggest_int('num_leaves', 8, 128),
            max_depth         = trial.suggest_int('max_depth', 3, 8),
            learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            min_child_samples = trial.suggest_int('min_child_samples', 5, 100),
            subsample         = trial.suggest_float('subsample', 0.4, 1.0),
            colsample_bytree  = trial.suggest_float('colsample_bytree', 0.4, 1.0),
            reg_alpha         = trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            reg_lambda        = trial.suggest_float('reg_lambda', 1e-3, 20.0, log=True),
            min_split_gain    = trial.suggest_float('min_split_gain', 0.0, 1.0),
        )
        m = lgb.LGBMClassifier(n_estimators=500, random_state=RANDOM_SEED, verbose=-1, **p)
        return temporal_cv_score(m, X, y, seasons)
    return _run_study(obj, n_trials, 'LightGBM')


def tune_xgb(X, y, seasons, n_trials=50):
    def obj(trial):
        p = dict(
            max_depth         = trial.suggest_int('max_depth', 3, 8),
            learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            subsample         = trial.suggest_float('subsample', 0.4, 1.0),
            colsample_bytree  = trial.suggest_float('colsample_bytree', 0.4, 1.0),
            gamma             = trial.suggest_float('gamma', 0.0, 5.0),
            min_child_weight  = trial.suggest_int('min_child_weight', 1, 20),
            reg_alpha         = trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            reg_lambda        = trial.suggest_float('reg_lambda', 1e-3, 20.0, log=True),
        )
        m = xgb.XGBClassifier(n_estimators=400, eval_metric='logloss',
                               random_state=RANDOM_SEED, verbosity=0, **p)
        return temporal_cv_score(m, X, y, seasons)
    return _run_study(obj, n_trials, 'XGBoost')


def tune_cat(X, y, seasons, n_trials=40):
    def obj(trial):
        p = dict(
            depth               = trial.suggest_int('depth', 3, 8),
            learning_rate       = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            l2_leaf_reg         = trial.suggest_float('l2_leaf_reg', 1.0, 50.0, log=True),
            bagging_temperature = trial.suggest_float('bagging_temperature', 0.0, 2.0),
            random_strength     = trial.suggest_float('random_strength', 0.0, 2.0),
            border_count        = trial.suggest_int('border_count', 32, 255),
        )
        m = CatBoostClassifier(iterations=400, random_seed=RANDOM_SEED, verbose=0, **p)
        return temporal_cv_score(m, X, y, seasons)
    return _run_study(obj, n_trials, 'CatBoost')


def tune_lr(X, y, seasons, n_trials=30):
    def obj(trial):
        C       = trial.suggest_float('C', 1e-3, 10.0, log=True)
        penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
        solver  = 'liblinear' if penalty == 'l1' else 'lbfgs'
        m = LogisticRegression(C=C, penalty=penalty, solver=solver, max_iter=2000)
        return temporal_cv_score(m, X, y, seasons)
    return _run_study(obj, n_trials, 'LogReg')


# ══════════════════════════════════════════════════════════════════════════════
#  8.  MAIN PREDICTOR
# ══════════════════════════════════════════════════════════════════════════════
class MarchMadnessPredictor:
    """Elite NCAA March Madness ensemble predictor."""

    def __init__(self, data_path):
        self.data_path  = data_path
        self.imputer    = SimpleImputer(strategy='median')
        self.scaler     = StandardScaler()
        self.col        = None
        self.sel_col    = None
        self.seeds      = None
        self.lgb_model  = None
        self.xgb_model  = None
        self.cat_model  = None
        self.lr_model   = None
        self.blend_weights = None
        self.calibrators = {}

    def load_data(self):
        """Load and engineer features from raw data."""
        # Load raw data
        data = load_all_data(self.data_path)
        
        # Extract seeds
        print("  Extracting seeds...")
        seeds_m = data.get('MNCAATourneySeeds')
        seeds_w = data.get('WNCAATourneySeeds')
        seeds_raw = pd.concat([seeds_m, seeds_w])
        self.seeds = {f"{int(r.Season)}_{int(r.TeamID)}": int(r.Seed[1:3])
                      for r in seeds_raw.itertuples()}
        print(f"  ✓ Extracted {len(self.seeds)} tournament seeds")

        # Load game data
        print("\n  Loading game data...")
        rs_m = data.get('MRegularSeasonDetailedResults')
        rs_w = data.get('WRegularSeasonDetailedResults')
        rs = pd.concat([rs_m, rs_w], ignore_index=True)
        
        tg_m = data.get('MNCAATourneyDetailedResults')
        tg_w = data.get('WNCAATourneyDetailedResults')
        tg = pd.concat([tg_m, tg_w], ignore_index=True)

        print("  Computing Elo ratings...")
        elo_df = compute_elo(rs)
        print("  Computing advanced team statistics...")
        stats_df = compute_team_stats(rs)
        print("  Computing head-to-head history...")
        h2h_df = compute_h2h(tg)

        print("  Building training features...")
        self.train_df = build_features(tg, self.seeds, elo_df, stats_df, is_train=True)
        self.train_df = self.train_df.merge(h2h_df, on=['T1', 'T2'], how='left')
        self.train_df['H2H_Games'] = self.train_df['H2H_Games'].fillna(0).astype(int)
        self.train_df['H2H_WinRate'] = self.train_df['H2H_WinRate'].fillna(0.5)

        # Get submission template
        sub_data = data.get('SampleSubmissionStage2')
        if sub_data is None:
            sub_data = data.get('SampleSubmissionStage1')
        
        print("  Building submission features...")
        sub_df_copy = sub_data.copy()
        self.sub_df  = build_features(sub_df_copy, self.seeds, elo_df, stats_df, is_train=False)
        self.sub_df = self.sub_df.merge(h2h_df, on=['T1', 'T2'], how='left')
        self.sub_df['H2H_Games'] = self.sub_df['H2H_Games'].fillna(0).astype(int)
        self.sub_df['H2H_WinRate'] = self.sub_df['H2H_WinRate'].fillna(0.5)
        
        self.sub_ids = sub_data['ID'].values

        EXCLUDE = {'Season', 'T1', 'T2', 'WTeamID', 'LTeamID', 'WScore', 'LScore',
                   'DayNum', 'NumOT', 'WLoc', 'ST', 'Label', 'Team1', 'Team2', 'ID',
                   'Pred', 'Conf1', 'Conf2', 'Seed', 'Seed1', 'Seed2',
                   'Season_str', 'T1_str', 'T2_str'}
        self.col = [c for c in self.train_df.columns
                    if c not in EXCLUDE
                    and pd.api.types.is_numeric_dtype(self.train_df[c])]
        print(f"\n  Raw features: {len(self.col)}")
        print("  Data loading complete.\n")

    def train_model(self):
        """Train ensemble with Bayesian tuning and isotonic calibration."""
        y       = self.train_df['Label'].values
        seasons = self.train_df['Season'].values

        # Impute & scale
        X_raw = self.train_df[self.col].values.astype(np.float32)
        X_raw = self.imputer.fit_transform(X_raw)
        X_raw = self.scaler.fit_transform(X_raw)

        # Feature selection
        print("=" * 80)
        print("  FEATURE SELECTION (Leakage-Safe)")
        print("=" * 80)
        self.sel_col = select_features(X_raw, y, self.col, seasons,
                                       n_top=80, held_out_seasons=5)
        X = self.train_df[self.sel_col].values.astype(np.float32)
        X = self.imputer.fit_transform(X)
        X = self.scaler.fit_transform(X)

        # Bayesian tuning
        print("\n" + "=" * 80)
        print("  BAYESIAN HYPERPARAMETER TUNING (Optuna TPE)")
        print("=" * 80)
        lgb_params, lgb_ll = tune_lgb(X, y, seasons, n_trials=60)
        xgb_params, xgb_ll = tune_xgb(X, y, seasons, n_trials=50)
        cat_params, cat_ll = tune_cat(X, y, seasons, n_trials=40)
        lr_params,  lr_ll  = tune_lr( X, y, seasons, n_trials=30)

        print(f"\n  Summary → LGB={lgb_ll:.5f}  XGB={xgb_ll:.5f}  "
              f"CAT={cat_ll:.5f}  LR={lr_ll:.5f}")

        # Build final models
        self.lgb_model = lgb.LGBMClassifier(
            n_estimators=1000, random_state=RANDOM_SEED, verbose=-1, **lgb_params)
        self.xgb_model = xgb.XGBClassifier(
            n_estimators=800, eval_metric='logloss',
            early_stopping_rounds=50, random_state=RANDOM_SEED,
            verbosity=0, **xgb_params)
        self.cat_model = CatBoostClassifier(
            iterations=800, random_seed=RANDOM_SEED, verbose=0, **cat_params)
        lr_pen    = lr_params.get('penalty', 'l2')
        lr_solver = 'liblinear' if lr_pen == 'l1' else 'lbfgs'
        self.lr_model = LogisticRegression(
            C=lr_params['C'], penalty=lr_pen, solver=lr_solver, max_iter=2000)

        # Temporal OOF
        print("\n" + "=" * 80)
        print("  TEMPORAL CROSS-VALIDATION & ISOTONIC CALIBRATION")
        print("=" * 80)
        unique_s = sorted(np.unique(seasons))
        oof = {m: np.full(len(y), np.nan) for m in ['lgb', 'xgb', 'cat', 'lr']}

        for ts in unique_s:
            tr = np.where(seasons < ts)[0]
            te = np.where(seasons == ts)[0]
            if len(tr) < 80 or len(te) == 0:
                continue
            Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]

            self.lgb_model.fit(
                Xtr, ytr,
                eval_set=[(Xte, yte)],
                callbacks=[lgb.early_stopping(60, verbose=False),
                           lgb.log_evaluation(period=-1)])
            oof['lgb'][te] = self.lgb_model.predict_proba(Xte)[:, 1]

            self.xgb_model.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)
            oof['xgb'][te] = self.xgb_model.predict_proba(Xte)[:, 1]

            self.cat_model.fit(Xtr, ytr, eval_set=(Xte, yte),
                               early_stopping_rounds=50)
            oof['cat'][te] = self.cat_model.predict_proba(Xte)[:, 1]

            self.lr_model.fit(Xtr, ytr)
            oof['lr'][te] = self.lr_model.predict_proba(Xte)[:, 1]

            print(f"  Season {ts:4d}  n={len(te):4d}  "
                  f"LGB_LL={log_loss(yte, oof['lgb'][te]):.5f}")

        valid = ~np.isnan(oof['lgb'])
        print(f"\n  OOF coverage: {valid.sum():,}/{len(y):,}")

        model_lls = {}
        for name, p in oof.items():
            ll = log_loss(y[valid], p[valid])
            model_lls[name] = ll
            print(f"  {name.upper()} OOF LL = {ll:.6f}")

        # Isotonic calibration
        print("\n  Fitting isotonic calibrators...")
        for name in ['lgb', 'xgb', 'cat', 'lr']:
            iso = IsotonicRegression(out_of_bounds='clip')
            iso.fit(oof[name][valid], y[valid])
            self.calibrators[name] = iso
            cal_preds = iso.predict(oof[name][valid])
            cal_ll    = log_loss(y[valid], np.clip(cal_preds, 1e-6, 1 - 1e-6))
            print(f"  {name.upper()} calibrated LL = {cal_ll:.6f}")

        # Weighted blend
        inv_ll = {k: 1.0 / v for k, v in model_lls.items()}
        total  = sum(inv_ll.values())
        self.blend_weights = {k: v / total for k, v in inv_ll.items()}
        print(f"\n  Blend weights: {' | '.join(f'{k.upper()}={v:.3f}' for k, v in self.blend_weights.items())}")

        # Final blend
        blend_oof = sum(
            self.blend_weights[m] * self.calibrators[m].predict(oof[m][valid])
            for m in ['lgb', 'xgb', 'cat', 'lr']
        )
        blend_oof = np.clip(blend_oof, 1e-6, 1 - 1e-6)
        final_cv_ll = log_loss(y[valid], blend_oof)
        print(f"  ⭐ FINAL BLEND OOF LL = {final_cv_ll:.6f}")

        # Retrain
        print("\n  Retraining all models on full data...")
        self.lgb_model.fit(X, y)
        self.xgb_model.set_params(early_stopping_rounds=None)
        self.xgb_model.fit(X, y)
        self.cat_model.fit(X, y)
        self.lr_model.fit(X, y)

        # Feature importance
        imp = pd.Series(self.lgb_model.feature_importances_,
                        index=self.sel_col).sort_values(ascending=False)
        print("\n  Top-15 Features:")
        for i, (feat, imp_val) in enumerate(imp.head(15).items(), 1):
            print(f"    {i:2d}. {feat:30s} {imp_val:.4f}")
        print("\n" + "=" * 80)
        print("  TRAINING COMPLETE")
        print("=" * 80 + "\n")

    def predict_submission(self, output_file='submission.csv'):
        """Generate final submission."""
        X_sub = self.sub_df[self.sel_col].values.astype(np.float32)
        X_sub = self.imputer.transform(X_sub)
        X_sub = self.scaler.transform(X_sub)

        raw = {
            'lgb': self.lgb_model.predict_proba(X_sub)[:, 1],
            'xgb': self.xgb_model.predict_proba(X_sub)[:, 1],
            'cat': self.cat_model.predict_proba(X_sub)[:, 1],
            'lr':  self.lr_model.predict_proba(X_sub)[:, 1],
        }

        cal = {m: np.clip(self.calibrators[m].predict(raw[m]), 1e-6, 1 - 1e-6)
               for m in raw}
        preds = sum(self.blend_weights[m] * cal[m] for m in cal)
        preds = np.clip(preds, 0.025, 0.975)

        out = pd.DataFrame({'ID': self.sub_ids, 'Pred': preds})
        out.to_csv(output_file, index=False)
        
        print("=" * 80)
        print(f"  SUBMISSION SAVED → {output_file}")
        print("=" * 80)
        print(f"  Predictions: n={len(preds):,}")
        print(f"  Mean prob: {preds.mean():.4f}")
        print(f"  Std prob:  {preds.std():.4f}")
        print(f"  Min prob:  {preds.min():.4f}")
        print(f"  Max prob:  {preds.max():.4f}")
        print("=" * 80 + "\n")
        
        return out


# ══════════════════════════════════════════════════════════════════════════════
#  ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == '__main__':
    print("\n" + "▪" * 80)
    print("▪" + " " * 78 + "▪")
    print("▪" + "NCAA MARCH MADNESS 2026 — ELITE ENSEMBLE PREDICTOR".center(78) + "▪")
    print("▪" + " " * 78 + "▪")
    print("▪" * 80 + "\n")
    
    try:
        predictor = MarchMadnessPredictor(DATA_PATH)
        predictor.load_data()
        predictor.train_model()
        predictor.predict_submission('submission.csv')
        print("\n✅ Pipeline complete! Ready for submission.\n")
    except Exception as e:
        print(f"\n❌ Error: {e}\n")
        import traceback
        traceback.print_exc()


▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪
▪                                                                              ▪
▪              NCAA MARCH MADNESS 2026 — ELITE ENSEMBLE PREDICTOR              ▪
▪                                                                              ▪
▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪

  LOADING DATA

  ✓ MNCAATourneySeeds                             (   2,694 rows)
  ✓ WNCAATourneySeeds                             (   1,812 rows)
  ✓ MRegularSeasonDetailedResults                 ( 124,529 rows)
  ✓ WRegularSeasonDetailedResults                 (  87,187 rows)
  ✓ MNCAATourneyDetailedResults                   (   1,449 rows)
  ✓ WNCAATourneyDetailedResults                   (     961 rows)
  ✓ SampleSubmissionStage2                        ( 132,133 rows)
  ✓ SampleSubmissionStage1                        ( 519,144 rows)

  Loaded 8 files

  Extracting seeds...
  ✓ Extr